# HDFS Log Preprocessing

### Objective

To transform raw HDFS log data into structured event sequences suitable for deep learning-based anomaly detection.

This notebook performs the following tasks:

- Parse raw HDFS log entries
- Extract important log fields
- Identify Block IDs
- Generate structured event sequences
- Merge sequences with anomaly labels
- Save the processed dataset for model training


## Import Libraries

In [1]:
import re
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

## Data Ingestion

In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')

DATA_PATH = "/content/drive/MyDrive/HDFS-Log-Analytics/data/raw"
LOG_FILE = f"{DATA_PATH}/HDFS.log"
LABEL_FILE = f"{DATA_PATH}/anomaly_label.csv"

for file in os.listdir(DATA_PATH):
    print(file)

Mounted at /content/drive
HDFS.log
anomaly_label.csv


## Anomaly Label Data Exploration

In [3]:
labels = pd.read_csv(LABEL_FILE)
print("Label Dataset Shape :", labels.shape)
labels.head(5)

Label Dataset Shape : (575061, 2)


,BlockId,Label
0,blk_-1608999687919862906,Normal
1,blk_7503483334202473044,Normal
2,blk_-3544583377289625738,Anomaly
3,blk_-9073992586687739851,Normal
4,blk_7854771516489510256,Normal


## HDFS Log Data Exploration

In [4]:
# Print first 5 lines of the Log
with open(LOG_FILE, "r") as file:
    for i in range(5):
        print(file.readline().strip())

081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
081109 203518 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.allocateBlock: /mnt/hadoop/mapred/system/job_200811092030_0001/job.jar. blk_-1608999687919862906
081109 203519 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.10.6:40524 dest: /10.250.10.6:50010
081109 203519 145 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.14.224:42420 dest: /10.250.14.224:50010
081109 203519 145 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_-1608999687919862906 terminating


## Parsing Raw HDFS Logs

In [5]:
# Extracting log fields from each logs
log_records = []

with open(LOG_FILE, "r") as file:
    for line in file:
        line = line.strip()
        if not line:
            continue
        parts = line.split()
        if len(parts) < 6:
            continue
        date = parts[0]
        time = parts[1]
        pid = parts[2]
        level = parts[3]
        component = parts[4].rstrip(":")
        message = " ".join(parts[5:])
        block_match = re.search(r"(blk_-?\d+)", line)
        block_id = block_match.group(1) if block_match else None
        log_records.append({
            "Date": date,
            "Time": time,
            "PID": pid,
            "Level": level,
            "Component": component,
            "Message": message,
            "BlockId": block_id
        })

logs_df = pd.DataFrame(log_records)

## Parsed Dataset Exploration

In [6]:
logs_df.sample(5, random_state=42)

,Date,Time,PID,Level,Component,Message,BlockId
4237366,081110,210118,28,INFO,dfs.FSNamesystem,BLOCK* NameSystem.delete: blk_-170857579237596...,blk_-1708575792375965353
7820301,081111,053144,21225,INFO,dfs.DataNode$PacketResponder,PacketResponder 2 for block blk_-3552845605773...,blk_-3552845605773916309
7694398,081111,051549,33,INFO,dfs.FSNamesystem,BLOCK* NameSystem.addStoredBlock: blockMap upd...,blk_-89648285413489710
536208,081109,220522,3276,WARN,dfs.DataNode$DataXceiver,10.250.14.38:50010:Got exception while serving...,blk_1817732267541489826
3596452,081110,132846,13,INFO,dfs.DataBlockScanner,Verification succeeded for blk_338667424734483...,blk_3386674247344831137


In [7]:
logs_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11175629 entries, 0 to 11175628
Data columns (total 7 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   Date       object
 1   Time       object
 2   PID        object
 3   Level      object
 4   Component  object
 5   Message    object
 6   BlockId    object
dtypes: object(7)
memory usage: 596.8+ MB


In [8]:
print("Total Log Entries :", len(logs_df))
print("Unique Block IDs :", logs_df["BlockId"].nunique())
print("Missing Block IDs :", logs_df["BlockId"].isna().sum())

Total Log Entries : 11175629
Unique Block IDs : 575061
Missing Block IDs : 0


In [9]:
print("Missing Values")
print(logs_df.isnull().sum())
print("\nDuplicate Rows :", logs_df.duplicated().sum())

Missing Values
Date         0
Time         0
PID          0
Level        0
Component    0
Message      0
BlockId      0
dtype: int64

Duplicate Rows : 2118


## Log Message Normalization

In [10]:
def normalize_message(message):
    message = re.sub(r"blk_-?\d+", "<BLOCK>", message) # Block IDs
    message = re.sub(r"\d+\.\d+\.\d+\.\d+(?::\d+)?", "<IP>", message) # IP addresses (with optional port)
    message = re.sub(r"/[\w./:-]+", "<PATH>", message) # File paths
    message = re.sub(r"\b\d+\b", "<NUM>", message)  # Standalone numbers
    return message

logs_df["Message"] = logs_df["Message"].apply(normalize_message)

In [11]:
logs_df["Message"].head(10)

,Message
0,Receiving block <BLOCK> src: /<IP> dest: /<IP>
1,BLOCK* NameSystem.allocateBlock: <PATH> <BLOCK>
2,Receiving block <BLOCK> src: /<IP> dest: /<IP>
3,Receiving block <BLOCK> src: /<IP> dest: /<IP>
4,PacketResponder <NUM> for block <BLOCK> termin...
5,PacketResponder <NUM> for block <BLOCK> termin...
6,Received block <BLOCK> of size <NUM> from /<IP>
7,Received block <BLOCK> of size <NUM> from /<IP>
8,PacketResponder <NUM> for block <BLOCK> termin...
9,Received block <BLOCK> of size <NUM> from /<IP>


## Event Template Generation

In [12]:
# Get all unique normalized messages
unique_events = sorted(logs_df["Message"].unique())

# Assign Event IDs
event_mapping = {event: f"E{i+1}" for i, event in enumerate(unique_events)}

print(f"Total Unique Event Templates : {len(event_mapping)}")

Total Unique Event Templates : 54


In [13]:
# Save event mapping as json for future use
import json

with open(
    "/content/drive/MyDrive/HDFS-Log-Analytics/data/processed/event_mapping.json",
    "w"
) as f:
    json.dump(event_mapping, f, indent=4)

In [14]:
# Convert Event Mapping to DataFrame
event_templates = pd.DataFrame({
    "EventId": list(event_mapping.values()),
    "EventTemplate": list(event_mapping.keys())
})

event_templates.head(10)

,EventId,EventTemplate
0,E1,<IP> Served block <BLOCK> to /<IP>
1,E2,<IP> Starting thread to transfer block <BLOCK>...
2,E3,<IP> Starting thread to transfer block <BLOCK>...
3,E4,<IP>:Exception writing block <BLOCK> to mirror...
4,E5,<IP>:Failed to transfer <BLOCK> to <IP> got ja...
5,E6,<IP>:Got exception while serving <BLOCK> to /<...
6,E7,<IP>:Transmitted block <BLOCK> to /<IP>
7,E8,Adding an already existing block <BLOCK>
8,E9,BLOCK* NameSystem.addStoredBlock: Redundant ad...
9,E10,BLOCK* NameSystem.addStoredBlock: addStoredBlo...


In [15]:
# Save event template as csv file for future use
output_dir = "/content/drive/MyDrive/HDFS-Log-Analytics/data/processed"
os.makedirs(output_dir, exist_ok=True)

event_templates.to_csv(
    f"{output_dir}/event_templates.csv",
    index=False
)

In [16]:
# Add EventId to logs_df
logs_df["EventId"] = logs_df["Message"].map(event_mapping)
logs_df.head()

,Date,Time,PID,Level,Component,Message,BlockId,EventId
0,081109,203518,143,INFO,dfs.DataNode$DataXceiver,Receiving block <BLOCK> src: /<IP> dest: /<IP>,blk_-1608999687919862906,E38
1,081109,203518,35,INFO,dfs.FSNamesystem,BLOCK* NameSystem.allocateBlock: <PATH> <BLOCK>,blk_-1608999687919862906,E12
2,081109,203519,143,INFO,dfs.DataNode$DataXceiver,Receiving block <BLOCK> src: /<IP> dest: /<IP>,blk_-1608999687919862906,E38
3,081109,203519,145,INFO,dfs.DataNode$DataXceiver,Receiving block <BLOCK> src: /<IP> dest: /<IP>,blk_-1608999687919862906,E38
4,081109,203519,145,INFO,dfs.DataNode$PacketResponder,PacketResponder <NUM> for block <BLOCK> termin...,blk_-1608999687919862906,E34


In [17]:
print("Event Template Statistics:")

print(f"Total Log Entries        : {len(logs_df):,}")
print(f"Unique Event Templates   : {len(event_templates):,}")
print(f"Unique Event IDs         : {logs_df['EventId'].nunique():,}")

Event Template Statistics:
Total Log Entries        : 11,175,629
Unique Event Templates   : 54
Unique Event IDs         : 54


## Event Sequence Generation

In [18]:
import gc

# Keep only the columns needed for sequence generation
logs_df = logs_df[["BlockId", "EventId"]]

# Remove rows without BlockId
logs_df = logs_df.dropna(subset=["BlockId"])

# Convert to category to reduce memory usage
logs_df["BlockId"] = logs_df["BlockId"].astype("category")
logs_df["EventId"] = logs_df["EventId"].astype("category")

gc.collect()

print(logs_df.info(memory_usage="deep"))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11175629 entries, 0 to 11175628
Data columns (total 2 columns):
 #   Column   Dtype   
---  ------   -----   
 0   BlockId  category
 1   EventId  category
dtypes: category(2)
memory usage: 109.1 MB
None


In [19]:
logs_df.head()

,BlockId,EventId
0,blk_-1608999687919862906,E38
1,blk_-1608999687919862906,E12
2,blk_-1608999687919862906,E38
3,blk_-1608999687919862906,E38
4,blk_-1608999687919862906,E34


In [20]:
# Group events by Block ID
event_sequences = (logs_df.groupby("BlockId", observed=True)["EventId"].apply(list).reset_index())
event_sequences.head()

,BlockId,EventId
0,blk_-1000002529962039464,"[E38, E38, E38, E12, E34, E36, E34, E36, E11, ..."
1,blk_-100000266894974466,"[E12, E38, E38, E38, E11, E11, E11, E34, E36, ..."
2,blk_-1000007292892887521,"[E38, E38, E12, E38, E34, E36, E34, E36, E34, ..."
3,blk_-1000014584150379967,"[E38, E12, E38, E38, E11, E11, E11, E34, E36, ..."
4,blk_-1000028658773048709,"[E38, E38, E38, E12, E34, E36, E34, E36, E34, ..."


In [21]:
# Rename Event ID to EventSequences
event_sequences.rename(columns={"EventId": "EventSequence"}, inplace=True)
event_sequences.head()

,BlockId,EventSequence
0,blk_-1000002529962039464,"[E38, E38, E38, E12, E34, E36, E34, E36, E11, ..."
1,blk_-100000266894974466,"[E12, E38, E38, E38, E11, E11, E11, E34, E36, ..."
2,blk_-1000007292892887521,"[E38, E38, E12, E38, E34, E36, E34, E36, E34, ..."
3,blk_-1000014584150379967,"[E38, E12, E38, E38, E11, E11, E11, E34, E36, ..."
4,blk_-1000028658773048709,"[E38, E38, E38, E12, E34, E36, E34, E36, E34, ..."


In [22]:
# Save Event Sequence as csv file
event_sequences.to_csv("/content/drive/MyDrive/HDFS-Log-Analytics/data/processed/event_sequences.csv", index=False)

In [23]:
# Add feature SequenceLength
event_sequences["SequenceLength"] = (event_sequences["EventSequence"].apply(len))
event_sequences.head()

,BlockId,EventSequence,SequenceLength
0,blk_-1000002529962039464,"[E38, E38, E38, E12, E34, E36, E34, E36, E11, ...",13
1,blk_-100000266894974466,"[E12, E38, E38, E38, E11, E11, E11, E34, E36, ...",28
2,blk_-1000007292892887521,"[E38, E38, E12, E38, E34, E36, E34, E36, E34, ...",13
3,blk_-1000014584150379967,"[E38, E12, E38, E38, E11, E11, E11, E34, E36, ...",29
4,blk_-1000028658773048709,"[E38, E38, E38, E12, E34, E36, E34, E36, E34, ...",19


In [24]:
print("Sequence Statistics:")
print(f"Total Sequences : {len(event_sequences):,}")
print(f"Average Length  : {event_sequences['SequenceLength'].mean():.2f}")
print(f"Maximum Length  : {event_sequences['SequenceLength'].max()}")
print(f"Minimum Length  : {event_sequences['SequenceLength'].min()}")

Sequence Statistics:
Total Sequences : 575,061
Average Length  : 19.43
Maximum Length  : 298
Minimum Length  : 2


## Final Pre-processed Dataset

In [25]:
# Merge Labels to Dataset
processed_df = event_sequences.merge(labels, on="BlockId", how="left")
processed_df.head()

,BlockId,EventSequence,SequenceLength,Label
0,blk_-1000002529962039464,"[E38, E38, E38, E12, E34, E36, E34, E36, E11, ...",13,Normal
1,blk_-100000266894974466,"[E12, E38, E38, E38, E11, E11, E11, E34, E36, ...",28,Normal
2,blk_-1000007292892887521,"[E38, E38, E12, E38, E34, E36, E34, E36, E34, ...",13,Normal
3,blk_-1000014584150379967,"[E38, E12, E38, E38, E11, E11, E11, E34, E36, ...",29,Normal
4,blk_-1000028658773048709,"[E38, E38, E38, E12, E34, E36, E34, E36, E34, ...",19,Normal


In [26]:
# Convert Labels to 0 and 1
processed_df["Label"] = processed_df["Label"].map({
    "Normal": 0,
    "Anomaly": 1
})

processed_df.head()

,BlockId,EventSequence,SequenceLength,Label
0,blk_-1000002529962039464,"[E38, E38, E38, E12, E34, E36, E34, E36, E11, ...",13,0
1,blk_-100000266894974466,"[E12, E38, E38, E38, E11, E11, E11, E34, E36, ...",28,0
2,blk_-1000007292892887521,"[E38, E38, E12, E38, E34, E36, E34, E36, E34, ...",13,0
3,blk_-1000014584150379967,"[E38, E12, E38, E38, E11, E11, E11, E34, E36, ...",29,0
4,blk_-1000028658773048709,"[E38, E38, E38, E12, E34, E36, E34, E36, E34, ...",19,0


In [27]:
# Save final processed dataset
processed_df.to_csv("/content/drive/MyDrive/HDFS-Log-Analytics/data/processed/processed_hdfs_dataset.csv", index=False)

In [31]:
print("Processed Dataset Summary:")
print(f"Rows               : {len(processed_df):,}")
print(f"Columns            : {processed_df.shape[1]}")
print(f"Normal Sequences   : {(processed_df['Label']==0).sum():,}")
print(f"Anomalous Sequences: {(processed_df['Label']==1).sum():,}")
print(f"Average Length     : {processed_df['SequenceLength'].mean():.2f}")

Processed Dataset Summary:
Rows               : 575,061
Columns            : 4
Normal Sequences   : 558,223
Anomalous Sequences: 16,838
Average Length     : 19.43


# Conclusion

The raw HDFS logs were successfully transformed into structured event sequences suitable for deep learning.

The preprocessing pipeline included:

- Parsing raw log entries
- Extracting Block IDs
- Normalizing log messages
- Generating event templates
- Creating ordered event sequences
- Merging with anomaly labels
- Encoding labels for model training
